In [1]:
# Personal parameters required by HW0/HW1 standing instructions
SID4 = 1384
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10
print({"SID4": SID4, "SEED": SEED, "SLICE": SLICE, "HP_ID": HP_ID, "CLS_A": CLS_A, "CLS_B": CLS_B})

{'SID4': 1384, 'SEED': 1384, 'SLICE': 384, 'HP_ID': 4, 'CLS_A': 4, 'CLS_B': 8}


# HW2.5 - GPU Assignment I

GPU: RTX 5090. This notebook must be executed on the reserved GPU workstation with outputs intact. Measurements are hardware-specific and must not be copied from another run.

In [2]:
from pathlib import Path
import shutil
import subprocess
import sys
import torch

ROOT = Path.cwd()
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
torch.manual_seed(SEED)
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

Python: 3.11.16 | packaged by conda-forge | (main, Sep  2 2026, 23:31:17) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.11.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5090


In [3]:
# Capture provenance before measuring. Preserve this output in the executed notebook.
smi = subprocess.run(['nvidia-smi', '-q'], capture_output=True, text=True, check=True)
print(smi.stdout)
(RESULTS / 'nvidia-smi-q.txt').write_text(smi.stdout, encoding='utf-8')


==============NVSMI LOG==============

Timestamp                                              : Tue Sep 15 18:54:18 2026
Driver Version                                         : 610.60 [Deprecated; will be removed in CUDA 14.0. Use KMD Version instead]
CUDA Version                                           : 13.3 [Deprecated; will be removed in CUDA 14.0. Use CUDA UMD Version instead]
KMD Version                                            : 610.60
CUDA UMD Version                                       : 13.3

Attached GPUs                                          : 1
GPU 00000000:01:00.0
    Product Name                                       : NVIDIA GeForce RTX 5090
    Product Brand                                      : GeForce
    Product Architecture                               : Blackwell
    Display Mode                                       : Requested functionality has been deprecated
    Display Attached                                   : Yes
    Display Active           

26557

In [4]:
# Run the required measurements. The thermal command intentionally runs for 20 minutes.
commands = [
    [sys.executable, 'benchmark_hw2_5.py', 'precision'],
    [sys.executable, 'benchmark_hw2_5.py', 'bandwidth'],
    [sys.executable, 'benchmark_hw2_5.py', 'attention'],
    [sys.executable, 'benchmark_hw2_5.py', 'thermal'],
    [sys.executable, 'benchmark_hw2_5.py', 'plot'],
]
for command in commands:
    print('RUN:', ' '.join(command))
    completed = subprocess.run(command, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        completed.check_returncode()

RUN: c:\Users\chelsi\Desktop\Chelsi\.venv\Scripts\python.exe benchmark_hw2_5.py precision

RUN: c:\Users\chelsi\Desktop\Chelsi\.venv\Scripts\python.exe benchmark_hw2_5.py bandwidth

RUN: c:\Users\chelsi\Desktop\Chelsi\.venv\Scripts\python.exe benchmark_hw2_5.py attention

RUN: c:\Users\chelsi\Desktop\Chelsi\.venv\Scripts\python.exe benchmark_hw2_5.py thermal

RUN: c:\Users\chelsi\Desktop\Chelsi\.venv\Scripts\python.exe benchmark_hw2_5.py plot



In [5]:
# Copy generated figures into the required figures/ directory.
for figure in RESULTS.glob('*.png'):
    shutil.copy2(figure, FIGURES / figure.name)
print('Figures:', sorted(path.name for path in FIGURES.glob('*')))
print('Raw results:', sorted(path.name for path in RESULTS.glob('*')))

Figures: ['attention_memory.png', 'precision_tflops.png', 'thermal_clock_temperature.png']
Raw results: ['attention.jsonl', 'attention_fit.txt', 'attention_memory.png', 'nvidia-smi-q.txt', 'precision.jsonl', 'precision_tflops.png', 'roofline.jsonl', 'thermal.csv', 'thermal_clock_temperature.png']


## Required interpretation

Complete `METRICS.md` from the UUID-labelled records after this notebook finishes. Include the measured precision plateaus, roofline classification, fitted attention coefficient, refined naive/fused OOM boundaries, fused speedups, and thermal throttle analysis. Record each command, timestamp, workstation, reservation ID, UUID, and GPU-hours in `RUN_LOG.txt`.